# Original LightGBM grouped five-fold reproduction
Only the original preprocessing and LightGBM grouped validation are executed. No searches or other models. Source cells are copied verbatim; model helper definitions are extracted verbatim by AST from `microplastics.py`. Historical unit assumptions are reproduced, not revalidated. All generated files remain in `reproduction_outputs/`.

In [1]:
import json, hashlib, platform, sys, importlib.metadata as metadata
from pathlib import Path
from collections.abc import Callable
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score
ROOT = Path.cwd()
OUT = ROOT / 'reproduction_outputs'
DATA = ROOT / 'data/noaa_microplastics.csv'
SEED = 0
pd.set_option('display.precision', 9)
environment = {'python': sys.version, 'platform': platform.platform(), 'packages': {p: metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','lightgbm','nbclient','ipykernel']}}
(OUT/'environment.json').write_text(json.dumps(environment, indent=2))
print(environment)

{'python': '3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ]', 'platform': 'macOS-15.6.1-arm64-arm-64bit', 'packages': {'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'lightgbm': '4.7.0', 'nbclient': '0.11.0', 'ipykernel': '7.4.0'}}


## Original loading, historical unit adjustments, features and folds
The original split cell also constructs random and chronological indices; neither is fitted or scored.

In [2]:
raw = pd.read_csv(DATA, low_memory=False)
data = raw.rename(columns={
    "Sample Date": "date",
    "Latitude (degree)": "lat",
    "Longitude (degree)": "lon",
    "Ocean": "ocean",
    "Marine Setting": "medium",
    "Water Sample Depth (m)": "depth",
    "Sampling Method": "method",
    "Mesh Size (mm)": "mesh",
    "Microplastics Measurement": "conc",
    "Unit": "unit",
    "NCEI Accession Number": "accession",
    "DOI": "doi",
    "Organization": "organization",
})[["date", "lat", "lon", "ocean", "medium", "depth", "method", "mesh", "conc", "unit", "accession", "doi", "organization"]]
data["date"] = pd.to_datetime(data.date, format="%m/%d/%Y %I:%M:%S %p")

print(f"{len(raw):,} records, {data.accession.nunique()} accessions, {data.organization.nunique()} organisations")
raw.isna().mean().sort_values(ascending=False).rename("missing").to_frame().T

29,076 records, 96 accessions, 68 organisations


,Transect Number,Ocean Bottom Depth (m),Sampling Point on Beach,Sediment Sample Depth (m),Subregion,State,Standardized Nurdle Amount,Volunteers Number,Collecting Time (min),Beach Location,...,Short Reference,Long Reference,DOI,Organization,Longitude (degree),NCEI Accession Number,NCEI Accession Link,ObjectId,Latitude (degree),Unique ID
missing,0.989750997,0.988581648,0.980740129,0.976613014,0.93661439,0.64706287,0.634922273,0.634922273,0.634922273,0.622919246,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
METHOD_FAMILY = {
    "neuston net": "surface net",
    "manta net": "surface net",
    "plankton net": "surface net",
    "avani net": "surface net",
    "bongo net": "surface net",
    "pump": "pump",
    "intake seawater pump": "pump",
    "surface water intake": "pump",
}
def linked_groups(a: pd.Series, b: pd.Series) -> pd.Series:
    """Labels rows linked through shared values of a or b, directly or via a chain, with one group."""
    group = a.astype(str)
    while not (merged := group.groupby(b).transform("min").groupby(a).transform("min")).equals(group):
        group = merged
    return group


UNIT_FIXES = {  # see section 1
    "https://doi.org/10.1016/j.envpol.2018.02.062": 1e6,  # Barrows et al. 2018
    "https://doi.org/10.1016/j.marpolbul.2020.111239": 1e6,  # Dehm et al. 2020
}
C = 1e-3  # offset for the log target, roughly the smallest non-zero concentration (6.8e-4)

water = data[(data.medium == "Ocean water") & (data.unit == "pieces/m3")].reset_index(drop=True)
water["conc"] *= water.doi.map(UNIT_FIXES).fillna(1)
method = water.method.str.lower().str.replace("-", " ")
family = method.map(METHOD_FAMILY).fillna("bulk water")
month = water.date.dt.month
df = water.assign(
    method=method,
    method_family=family,
    fam_pump=(family == "pump").astype(float),
    fam_bulk=(family == "bulk water").astype(float),
    year=water.date.dt.year,
    month_sin=np.sin(2 * np.pi * month / 12),
    month_cos=np.cos(2 * np.pi * month / 12),
    log_mesh=np.log10(water.mesh),
    log_depth=np.log1p(water.depth),
    study=linked_groups(water.doi, water.accession),
    y=np.log10(water.conc + C),
)
print(f"{len(df):,} records, {df.accession.nunique()} accessions, {df.doi.nunique()} publications, {df.study.nunique()} studies, {df.organization.nunique()} organisations")
df.groupby(["method_family", "method"]).agg(
    records=("y", "size"), accessions=("accession", "nunique"), median_conc=("conc", "median")
).sort_values(["method_family", "records"], ascending=[True, False])

16,988 records, 73 accessions, 73 publications, 72 studies, 52 organisations


records  accessions   median_conc
method_family method                                                    
bulk water    grab sample                 1389           1  5000.0000000
              van dorn sampler             181           1     0.0000000
              niskin bottle                 81           1  1300.0000000
              stainless steel bucket        80           2   860.0000000
              aluminum bucket               57           1  1710.0000000
              ctd rosette sampler           48           1    21.0000000
              steel bucket                  43           3   430.0000000
              stainless steel sampler       23           1  2600.0000000
              glass jar                      1           1     0.0180000
pump          pump                         563           1     6.0000000
              intake seawater pump         284           4     1.0000000
              surface water intake           5           1  5015.0000000
surface net   neuston net                10720          17     0.0038800
              manta net                   3204          36     0.3627245
              plankton net                 240           8     0.4315000
              avani net                     36           1     0.0625370
              bongo net                     33           1  2860.0000000

In [4]:
NUMERIC = ["lat", "lon", "year", "month_sin", "month_cos", "log_mesh", "log_depth"]
CATEGORICAL = ["method_family"]
FEATURES = NUMERIC + CATEGORICAL
CUTOFF = 2015

SPLITS = {
    "random": list(KFold(5, shuffle=True, random_state=SEED).split(df)),
    "grouped": list(GroupKFold(5).split(df, groups=df.study)),
    "chrono": [(np.flatnonzero(df.year < CUTOFF), np.flatnonzero(df.year >= CUTOFF))],
}
for scheme, folds in SPLITS.items():
    sizes = [len(test) for _, test in folds]
    print(f"{scheme:>8}: {len(folds)} fold(s), test sizes {sizes}")
pre, post = df[df.year < CUTOFF], df[df.year >= CUTOFF]
seen = post.study.isin(pre.study)
print(f"chrono: {pre.study.nunique()} studies before {CUTOFF}, {post.study.nunique()} after, "
      f"{post[seen].study.nunique()} in both. {seen.sum():,} of {len(post):,} test rows come from studies seen in training")

  random: 5 fold(s), test sizes [3398, 3398, 3398, 3397, 3397]
 grouped: 5 fold(s), test sizes [6135, 2714, 2712, 2714, 2713]
  chrono: 1 fold(s), test sizes [4638]
chrono: 25 studies before 2015, 54 after, 7 in both. 2,155 of 4,638 test rows come from studies seen in training


In [5]:
Predictor = Callable[[pd.DataFrame], np.ndarray]

Model = Callable[[pd.DataFrame], Predictor]

def preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])

def clip_to_range(train: pd.DataFrame) -> Callable[[pd.DataFrame], pd.DataFrame]:
    """Clips numeric features to the training range, so models that extrapolate (linear, splines) stay flat
    outside it. Trees already behave this way."""
    low, high = train[NUMERIC].min(), train[NUMERIC].max()
    return lambda frame: frame.assign(**{c: frame[c].clip(low[c], high[c]) for c in NUMERIC})

def sklearn_model(estimator) -> Model:
    def fit(train: pd.DataFrame) -> Predictor:
        clip = clip_to_range(train)
        pipe = make_pipeline(preprocessor(), clone(estimator)).fit(train[FEATURES], train.y)
        return lambda test: pipe.predict(clip(test)[FEATURES])
    return fit

BOOSTER_DEFAULTS = dict(n_estimators=500, learning_rate=0.05, random_state=SEED, n_jobs=4)

def metrics(y: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    err = pred - y
    raw_err = np.clip(10**pred - C, 0, None) - (10**y - C)
    return {
        "R2": 1 - np.sum(err**2) / np.sum((y - y.mean()) ** 2),
        "MAE": np.mean(np.abs(err)),
        "RMSE": np.sqrt(np.mean(err**2)),
        "within 1 log": np.mean(np.abs(err) <= 1),
        "raw MAE": np.mean(np.abs(raw_err)),
        "raw RMSE": np.sqrt(np.mean(raw_err**2)),
    }

def score(pred: pd.Series) -> dict[str, float]:
    """Metrics for a series of out-of-fold predictions indexed like df. NaN entries are skipped."""
    pred = pred.dropna()
    y = df.y.loc[pred.index]
    study_mae = (pred - y).abs().groupby(df.study.loc[pred.index]).mean().mean()
    return metrics(y.to_numpy(), pred.to_numpy()) | {"study MAE": study_mae}

def out_of_fold(model: Model, scheme: str) -> pd.Series:
    pred = pd.Series(np.nan, index=df.index)
    for train_idx, test_idx in SPLITS[scheme]:
        pred.iloc[test_idx] = model(df.iloc[train_idx])(df.iloc[test_idx])
    return pred

model = sklearn_model(LGBMRegressor(**BOOSTER_DEFAULTS, verbose=-1))
(OUT/"model_parameters.json").write_text(json.dumps(LGBMRegressor(**BOOSTER_DEFAULTS, verbose=-1).get_params(), indent=2))

467

## Focused integrity and preprocessing tests

In [6]:
checks = []
def check(name, condition):
    assert condition, name
    checks.append(name)
check('raw shape matches saved notebook', raw.shape == (29076,35))
check('modelling counts match saved notebook', (len(df),df.accession.nunique(),df.doi.nunique(),df.study.nunique()) == (16988,73,73,72))
check('finite target; numeric features contain no infinity', np.isfinite(df.y).all() and not np.isinf(df[NUMERIC].to_numpy()).any())
base = data[(data.medium == 'Ocean water') & (data.unit == 'pieces/m3')].reset_index(drop=True)
check('historical scaling exact', np.array_equal(df.conc.to_numpy(), (base.conc * base.doi.map(UNIT_FIXES).fillna(1)).to_numpy()))
check('log target exact', np.array_equal(df.y.to_numpy(), np.log10(df.conc.to_numpy()+C)))
# Independent union-find verifies connected components of DOI/accession links.
parent = list(range(len(base)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for column in ['doi','accession']:
    for indices in base.groupby(column).indices.values():
        for i in indices[1:]: parent[root(int(i))] = root(int(indices[0]))
components = pd.Series([root(i) for i in range(len(base))])
check('study grouping equals independent connected components', components.groupby(df.study).nunique().max()==1 and df.study.groupby(components).nunique().max()==1)
fold_id = np.zeros(len(df),dtype=int)
check('original fold sizes', [len(t) for _,t in SPLITS['grouped']] == [6135,2714,2712,2714,2713])
for f,(tr,te) in enumerate(SPLITS['grouped'],1):
    check(f'fold {f} no row or study leakage', not set(tr)&set(te) and not set(df.study.iloc[tr])&set(df.study.iloc[te]) and len(tr)+len(te)==len(df))
    check(f'fold {f} no DOI/accession leakage', all(not set(df[c].iloc[tr])&set(df[c].iloc[te]) for c in ['doi','accession']))
    check(f'fold {f} unique test assignment', (fold_id[te]==0).all())
    fold_id[te]=f
    prep=preprocessor().fit(df.iloc[tr][FEATURES])
    check(f'fold {f} training-only median imputation', np.allclose(prep.named_transformers_['num'].named_steps['simpleimputer'].statistics_, df.iloc[tr][NUMERIC].median()))
check('every row tested once', (fold_id>0).all())
unit_audit = [{'doi': doi, 'multiplier': factor, 'rows': int((base.doi==doi).sum()), 'raw_mean': float(base.loc[base.doi==doi,'conc'].mean()), 'adjusted_mean': float(df.loc[df.doi==doi,'conc'].mean())} for doi,factor in UNIT_FIXES.items()]
(OUT/'unit_adjustments.json').write_text(json.dumps(unit_audit, indent=2))
print(unit_audit)
print(f'{len(checks)} preprocessing tests passed')

[{'doi': 'https://doi.org/10.1016/j.envpol.2018.02.062', 'multiplier': 1000000.0, 'rows': 1389, 'raw_mean': 0.012777537796976242, 'adjusted_mean': 12777.537796976241}, {'doi': 'https://doi.org/10.1016/j.marpolbul.2020.111239', 'multiplier': 1000000.0, 'rows': 81, 'raw_mean': 0.0017629629629629627, 'adjusted_mean': 1762.962962962963}]
28 preprocessing tests passed


## Execute the unchanged original out-of-fold helper (five LightGBM fits)

In [7]:
pred = out_of_fold(model, 'grouped')
check('finite complete predictions', np.isfinite(pred).all() and len(pred)==len(df))
pooled = score(pred)
check('R2 agrees with sklearn', np.isclose(pooled['R2'], r2_score(df.y,pred), atol=1e-14))
fold_metrics = pd.DataFrame([{'fold': f, 'train_rows':len(tr), 'test_rows':len(te), 'train_studies':df.study.iloc[tr].nunique(), 'test_studies':df.study.iloc[te].nunique(), **score(pred.iloc[te])} for f,(tr,te) in enumerate(SPLITS['grouped'],1)])
fold_metrics.to_csv(OUT/'fold_metrics.csv',index=False)
pd.DataFrame({'water_row':df.index, 'source_row':data.index[(data.medium=='Ocean water') & (data.unit=='pieces/m3')], 'study':df.study, 'doi':df.doi, 'accession':df.accession, 'fold':fold_id, 'target_log10':df.y, 'prediction_log10':pred}).to_csv(OUT/'out_of_fold_predictions.csv',index=False)
summary = {'raw_rows':len(raw), 'raw_columns':len(raw.columns), 'modelling_rows':len(df), 'studies':df.study.nunique(), 'accessions':df.accession.nunique(), 'publications':df.doi.nunique(), 'pooled':pooled, 'fold_mean_R2':float(fold_metrics.R2.mean()), 'reported_R2':0.473, 'difference_from_reported':float(pooled['R2']-0.473), 'matches_reported_to_3_decimals':round(float(pooled['R2']),3)==0.473}
(OUT/'summary.json').write_text(json.dumps(summary,indent=2,default=float))
print(json.dumps(summary,indent=2,default=float))
display(fold_metrics)


{
  "raw_rows": 29076,
  "raw_columns": 35,
  "modelling_rows": 16988,
  "studies": 72,
  "accessions": 73,
  "publications": 73,
  "pooled": {
    "R2": 0.4726308398591973,
    "MAE": 0.9963774232101922,
    "RMSE": 1.418197730643938,
    "within 1 log": 0.639392512361667,
    "raw MAE": 1093.9747133366286,
    "raw RMSE": 8638.33742300612,
    "study MAE": 1.0609421643148442
  },
  "fold_mean_R2": -0.07123499211367457,
  "reported_R2": 0.473,
  "difference_from_reported": -0.00036916014080268233,
  "matches_reported_to_3_decimals": true
}


,fold,train_rows,test_rows,train_studies,test_studies,R2,MAE,RMSE,within 1 log,raw MAE,raw RMSE,study MAE
0,1,10853,6135,71,1,-0.555995934,0.727918876,0.909011644,0.724205379,0.029770640,0.098199835,0.727918876
1,2,14274,2714,61,11,0.003501738,0.812758445,1.003800108,0.690493736,28.152624882,1123.526150688,0.866408622
2,3,14276,2712,52,20,0.301706051,1.602671579,2.273102127,0.483407080,6455.379597326,21399.195077995,1.220970668
3,4,14274,2714,52,20,-0.437682635,1.275470188,1.739840234,0.526529108,213.576485965,749.251025963,1.041255160
4,5,14275,2713,52,20,0.332295820,0.901872291,1.234620532,0.665315149,155.258943053,2769.949369974,1.044245277


In [8]:
before = json.loads((OUT/'input_hashes.json').read_text())
check('all original tracked files unchanged', all(hashlib.sha256((ROOT/p).read_bytes()).hexdigest()==h for p,h in before.items()))
(OUT/'tests.json').write_text(json.dumps({'passed':len(checks),'checks':checks},indent=2))
print(f'{len(checks)} focused tests passed; source and dataset unchanged.')

31 focused tests passed; source and dataset unchanged.
